# 13 · Proves, depuració i integració

**FES · UAB · 2026–2027 · 1 hora de teoria**

En acabar podràs expressar un resultat esperat amb `assert`, escriure proves amb `unittest`, seleccionar casos límit i comprovar el recorregut complet d'una mesura fins a un fitxer.

Treballarem amb els components facilitats dels blocs 10 i 12. Les primeres proves verifiquen un `Sensor`. Després combinarem `SistemaMonitoratge` amb CSV i JSON. Executa totes les cel·les en ordre des d'un kernel net.

## Recuperar els components del projecte

Un component reutilitzable té una responsabilitat concreta. El sensor gestiona lectures, el sistema agrupa sensors i les funcions de persistència desen i recuperen dades.

Els noms docents comencen amb números. `importlib.import_module()` rep el nom del mòdul com a cadena i permet recuperar aquestes definicions. L'adaptador següent ja està preparat. Pots obrir el notebook des de l'arrel del repositori o des de la carpeta del bloc 13. Els components són els exemples resolts de teoria, disponibles encara que no hagis fet els exercicis.

In [ ]:
import importlib
from pathlib import Path
import sys
from tempfile import TemporaryDirectory
import unittest

ARREL = Path.cwd()
if ARREL.name == "13_proves_integracio":
    ARREL = ARREL.parent
if str(ARREL) not in sys.path:
    sys.path.insert(0, str(ARREL))

modul_disseny = importlib.import_module(
    "12_poo_disseny.01_sistema_monitoratge"
)
modul_csv = importlib.import_module("10_fitxers.02_csv")
modul_json = importlib.import_module("10_fitxers.03_json")
Sensor = modul_disseny.Sensor
SistemaMonitoratge = modul_disseny.SistemaMonitoratge

## Una expectativa amb `assert`

`assert condicio, "missatge"` genera `AssertionError` quan la condició és falsa. Calcula a mà quants mil·livolts corresponen a 1.2 V abans d'executar el codi.

L'entrada zero també ha de funcionar. Amb l'opció `python -O` es poden desactivar els `assert`. La validació de dades que arriben al programa utilitza `if` i `raise`, com al bloc 09.

In [ ]:
def volts_a_milivolts(tensio):
    """Converteix una tensió numèrica de volts a mil·livolts."""
    return tensio * 1000


def demostrar_asserts():
    """Comprova un exemple conegut i el límit zero."""
    assert volts_a_milivolts(1.2) == 1200.0, "Conversió incorrecta"
    assert volts_a_milivolts(0) == 0, "El zero s'ha de conservar"
    print("Les dues comprovacions amb assert han passat.")

demostrar_asserts()

## Proves unitàries amb `unittest`

La biblioteca estàndard `unittest` agrupa comprovacions en classes derivades de `unittest.TestCase`. Cada mètode que comença amb `test_` és una prova. Aquest patró està donat i el reutilitzarem.

`assertEqual` compara resultats, `assertIsNone` comprova que encara no existeix una lectura i `assertAlmostEqual` compara decimals amb tolerància a l'arrodoniment. `with self.assertRaises(ValueError)` comprova que l'operació interior genera l'error esperat.

Cada prova crea les seves dades perquè es pugui executar independentment. Provar dues instàncies ajuda a detectar una llista de lectures compartida accidentalment. Prediu què haurà de conservar el sensor quan un registre falla.

In [ ]:
class TestSensor(unittest.TestCase):
    """Proves unitàries del comportament d'un únic sensor."""

    def test_sensor_sense_lectures(self):
        sensor = Sensor("T1", "°C")
        self.assertIsNone(sensor.ultima_lectura())
        self.assertIsNone(sensor.mitjana())

    def test_registre_i_mitjana(self):
        sensor = Sensor("V1", "V")
        sensor.registrar(0.1)
        sensor.registrar(0.2)
        self.assertEqual(sensor.ultima_lectura(), 0.2)
        self.assertAlmostEqual(sensor.mitjana(), 0.15)

    def test_instancies_independents(self):
        primer = Sensor("T1", "°C")
        segon = Sensor("T2", "°C")
        primer.registrar(22.5)
        self.assertIsNone(segon.ultima_lectura())

    def test_valor_invalid_conserva_estat(self):
        sensor = Sensor("T1", "°C")
        sensor.registrar(22.5)
        with self.assertRaises(ValueError):
            sensor.registrar("desconnectat")
        self.assertEqual(sensor.ultima_lectura(), 22.5)

## Integrar el sistema i la persistència

Una prova d'integració verifica el recorregut de la dada entre components. Crearem un sistema, afegirem un sensor, registrarem una lectura, desarem el resum i recuperarem el contingut.

`TemporaryDirectory` ofereix una carpeta nova per a cada prova i elimina els fitxers en sortir del `with`. Comparem el registre complet per verificar que es conserven nom, unitat i valor. Les dades d'aquestes proves tenen una lectura abans de l'exportació.

Un sensor sense lectura aporta `None` al resum. Les funcions de persistència del bloc 10 esperen valors numèrics finits. `lectures_disponibles()` selecciona les files amb lectura abans d'exportar-les. La comprovació `is not None` conserva també el zero. Afegim proves per a un sistema buit i per a un sistema mixt amb un sensor pendent i un altre amb lectura zero, en cadascun dels formats.

In [ ]:
def lectures_disponibles(sistema):
    """Retorna només les files del resum que ja tenen una lectura."""
    disponibles = []
    for fila in sistema.resum():
        if fila["valor"] is not None:
            disponibles.append(fila)
    return disponibles


class TestIntegracio(unittest.TestCase):
    """Proves del recorregut sensor → sistema → fitxer → dades."""

    def test_recorregut_csv(self):
        sistema = SistemaMonitoratge()
        sistema.afegir_sensor(Sensor("T1", "°C"))
        sistema.registrar("T1", 22.5)
        esperat = [{"sensor": "T1", "unitat": "°C", "valor": 22.5}]
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "lectures.csv"
            modul_csv.desar_csv(ruta, lectures_disponibles(sistema))
            recuperat = modul_csv.llegir_csv(ruta)
        self.assertEqual(recuperat, esperat)

    def test_recorregut_json(self):
        sistema = SistemaMonitoratge()
        sistema.afegir_sensor(Sensor("H1", "%"))
        sistema.registrar("H1", 48.0)
        esperat = [{"sensor": "H1", "unitat": "%", "valor": 48.0}]
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "lectures.json"
            modul_json.desar_json(ruta, lectures_disponibles(sistema))
            recuperat = modul_json.llegir_json(ruta)
        self.assertEqual(recuperat, esperat)

    def test_sistema_buit_csv(self):
        sistema = SistemaMonitoratge()
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "buit.csv"
            modul_csv.desar_csv(ruta, lectures_disponibles(sistema))
            recuperat = modul_csv.llegir_csv(ruta)
        self.assertEqual(recuperat, [])

    def test_sistema_buit_json(self):
        sistema = SistemaMonitoratge()
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "buit.json"
            modul_json.desar_json(ruta, lectures_disponibles(sistema))
            recuperat = modul_json.llegir_json(ruta)
        self.assertEqual(recuperat, [])

    def test_sistema_mixt_csv(self):
        sistema = SistemaMonitoratge()
        sistema.afegir_sensor(Sensor("T1", "°C"))
        sistema.afegir_sensor(Sensor("V1", "V"))
        sistema.registrar("V1", 0)
        esperat = [{"sensor": "V1", "unitat": "V", "valor": 0.0}]
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "mixt.csv"
            modul_csv.desar_csv(ruta, lectures_disponibles(sistema))
            recuperat = modul_csv.llegir_csv(ruta)
        self.assertEqual(recuperat, esperat)

    def test_sistema_mixt_json(self):
        sistema = SistemaMonitoratge()
        sistema.afegir_sensor(Sensor("T1", "°C"))
        sistema.afegir_sensor(Sensor("V1", "V"))
        sistema.registrar("V1", 0)
        esperat = [{"sensor": "V1", "unitat": "V", "valor": 0.0}]
        with TemporaryDirectory() as directori:
            ruta = Path(directori) / "mixt.json"
            modul_json.desar_json(ruta, lectures_disponibles(sistema))
            recuperat = modul_json.llegir_json(ruta)
        self.assertEqual(recuperat, esperat)

## Executar i interpretar els resultats

`TestLoader` recull els mètodes de prova. `TestSuite` els agrupa i `TextTestRunner` mostra el resultat de cadascun. Aquest runner explícit funciona dins de Jupyter sense interpretar els arguments del kernel.

S'esperen **deu proves correctes** i el missatge `OK`. Un `FAIL` assenyala una expectativa que no s'ha complert. Un `ERROR` indica que una operació ha generat un error que la prova no esperava. Llegeix el nom del test i la línia indicada abans de canviar codi.

In [ ]:
carregador = unittest.TestLoader()
suite = unittest.TestSuite([
    carregador.loadTestsFromTestCase(TestSensor),
    carregador.loadTestsFromTestCase(TestIntegracio),
])
resultat = unittest.TextTestRunner(verbosity=2).run(suite)
if not resultat.wasSuccessful():
    raise AssertionError("Cal revisar les proves fallides")

## Depurar a partir d'un cas reproduïble

Una funció que divideix la suma de `[20, 24]` entre `len(lectures) + 1` retorna aproximadament 14.67. El resultat calculat a mà és 22.0.

1. Reprodueix la fallada amb aquestes dues lectures.
2. Col·loca un punt d'interrupció abans del retorn o imprimeix temporalment la suma i el divisor.
3. Inspecciona el nombre de lectures i corregeix la causa.
4. Executa de nou el cas original i afegeix proves per a `[]` i `[0]`.

Conservar aquests casos com a proves de regressió ajuda a mantenir el programa quan evoluciona.

### Pràctica

Resol els exercicis 1–3 com a contingut imprescindible. L'exercici 4 és un repte opcional de depuració.

[Enunciats](01_proves_integracio_exercicis.py) · [Solucions](01_proves_integracio_solucions.py)